# 18회. 문자열 다루기 — 접수번호·이름·기간 쪼개기

**오늘의 목표**
1. 문자열 **자르기**(슬라이싱)와 **나누기**(`split`)로 필요한 부분을 꺼낸다.
2. `strip`, `replace`, `upper`, `startswith` 등으로 **정리**한다.
3. 같은 것을 다르게 쓴 값(`CO2`, `이산화탄소`, `CO₂`)을 **통일**한다.
4. 정규표현식(`re`)으로 글 속의 **숫자·패턴**을 찾는다.

> 이 노트북의 문제들은 04회 "업무 속 문제 찾기"의 실제 사례(정보공개청구 출장기록, 실적 보고)를 가상 데이터로 바꾼 것입니다.

> ✅ **해답 노트북** — 연습 문제의 예시 답안이 채워져 있습니다.

## 1. 자르기 — 위치로 꺼내기
문자열도 리스트처럼 **번호(0부터)** 로 꺼낼 수 있습니다.
기존 R 코드 `실내공기 분기 보고서.R`의 `substr(접수번호, 1, 10)`이 바로 이 작업입니다.

In [ ]:
receipt = "2792400354-01"
print(receipt[:10])     # 앞 10자리 → 접수번호 본번호
print(receipt[-2:])     # 뒤 2자리 → 시료 번호
print(len(receipt))

## 2. 나누기 — split
**구분 기호**를 기준으로 나눠 **리스트**로 돌려줍니다. R의 `str_split()`, 엑셀의 "텍스트 나누기"와 같습니다.

In [ ]:
receipt = "2792400354-01"
base, sub = receipt.split("-")
print(base, sub)

sample_type = "실내공기질/다중이용시설/어린이집"
parts = sample_type.split("/")
print(parts)
print("시설군:", parts[2])        # R 코드의 sapply("[", 3) 와 같음 (R은 1부터, 파이썬은 0부터!)

### 🧪 연습 1 — 이름과 코드 분리 (04회 사례)
출장기록의 성명 칸이 `"홍길동(X0000001)"` 모양입니다. **이름**과 **코드**를 각각 꺼내 출력하세요.
(힌트: `"("`로 나누고, 코드 끝의 `")"`는 `replace`로 지우기)

In [ ]:
name_code = "홍길동(X0000001)"
name, code = name_code.split("(")
code = code.replace(")", "")
print("이름:", name)    # 홍길동
print("코드:", code)    # X0000001

### 🧪 연습 2 — 출장기간 4칸으로 (04회 사례)
`"2025-04-16 09:40 ~ 2025-04-16 11:50"` 을 **출발일, 출발시간, 도착일, 도착시간**으로 나누세요.
시간이 `"--:--"` 이면 빈 문자열 `""` 로 바꾸세요. 두 번째 예 `"2025-01-21 --:-- ~ 2025-01-21 --:--"`로도 확인하세요.

In [ ]:
period = "2025-04-16 09:40 ~ 2025-04-16 11:50"
start, end = period.split(" ~ ")
start_date, start_time = start.split(" ")
end_date, end_time = end.split(" ")
if start_time == "--:--":
    start_time = ""
if end_time == "--:--":
    end_time = ""
print([start_date, start_time, end_date, end_time])

## 3. 정리하기
| 메서드 | 하는 일 | 예 → 결과 |
|---|---|---|
| `strip()` | 앞뒤 공백 제거 | `"  CO2 "` → `"CO2"` |
| `replace(a, b)` | a를 b로 바꾸기 | `"대 기"` → `.replace(" ", "")` → `"대기"` |
| `upper()` / `lower()` | 대문자 / 소문자 | `"co2"` → `"CO2"` |
| `startswith(a)` / `endswith(a)` | a로 시작/끝나나 | `"2792..."`.startswith("279") → True |
| `a in s` | 들어 있나 | `"실내공기질" in sample_type` |
| `zfill(n)` | 앞을 0으로 채워 n자리 | `"7"` → `"007"` |
| `"/".join(리스트)` | 리스트를 이어 붙이기 | `["a","b"]` → `"a/b"` |

In [ ]:
raw = "  대기및악취/지도점검/대 기  "
clean = raw.strip().replace(" ", "")
print(f"[{raw}] → [{clean}]")

print("실내공기질" in "실내공기질/다중이용시설/어린이집")
print(str(7).zfill(3))
print(" > ".join(["실내공기질", "다중이용시설", "어린이집"]))

## 4. 같은 것을 다르게 쓴 값 통일하기
04회 실적 보고 사례의 방해 요소: 항목명이 `이산화탄소`, `CO2`, `CO₂`, `co2 ` 처럼 제각각.
→ **"표준 이름 사전"** 을 만들어 통일합니다.

In [ ]:
standard_names = {
    "이산화탄소": "CO2",
    "CO2": "CO2",
    "CO₂": "CO2",
    "폼알데하이드": "HCHO",
    "포름알데히드": "HCHO",
    "HCHO": "HCHO",
}

def normalize_item(name):
    """항목 이름의 공백·대소문자를 정리한 뒤 표준 이름으로 바꾼다. 사전에 없으면 그대로."""
    key = name.strip()
    if key.upper() in standard_names:      # 'co2' 같은 소문자도 처리
        key = key.upper()
    return standard_names.get(key, key)

for raw in ["이산화탄소", "CO₂", "co2 ", "포름알데히드", "라돈"]:
    print(f"{raw!r:>10} → {normalize_item(raw)}")

### 🧪 연습 3
시료명이 `"2025.04.08 효자동 1층"` 모양입니다. 03회 "예시 주기" 실습처럼 `"250408-효자-1F"` 모양으로 바꾸는 함수 `short_name(text)`를 만드세요.
- 날짜 `2025.04.08` → `250408` (점 지우고 앞 2자리 빼기)
- 장소 `효자동` → 앞 2글자 `효자`
- 층 `1층` → `1F`

`"2025.05.12 가평읍 3층"` → `"250512-가평-3F"` 도 확인하세요.

In [ ]:
def short_name(text):
    date, place, floor = text.split(" ")
    date = date.replace(".", "")[2:]
    place = place[:2]
    floor = floor.replace("층", "F")
    return f"{date}-{place}-{floor}"

print(short_name("2025.04.08 효자동 1층"))   # 250408-효자-1F
print(short_name("2025.05.12 가평읍 3층"))   # 250512-가평-3F

## 5. 정규표현식 맛보기 — 패턴으로 찾기
**정규표현식(regex)** 은 "숫자 여러 개", "숫자10자리-숫자2자리" 같은 **패턴**을 찾는 문법입니다.
처음부터 외울 필요는 없고, **AI에게 패턴을 만들어 달라고 한 뒤 테스트**하는 방식으로 씁니다.

| 패턴 | 뜻 |
|---|---|
| `\d` | 숫자 한 개 |
| `\d+` | 숫자 한 개 이상 |
| `\d{10}` | 숫자 정확히 10개 |
| `\.?` | 점이 있어도 되고 없어도 됨 |

In [ ]:
import re

memo = "4/8 효자동 어린이집 1층 CO2 평균 612, 2층 845.5 / 폼알 45.2"
numbers = re.findall(r"\d+\.?\d*", memo)
print(numbers)

text = "접수번호 2792400354-01, 2792400355-02 결과 송부"
print(re.findall(r"\d{10}-\d{2}", text))

> 첫 결과에 날짜(`4`, `8`)와 층(`1`, `2`)까지 섞여 나오는 것을 보세요. 패턴은 **생각보다 넓게** 잡히기 쉽습니다. 그래서 **결과를 꼭 눈으로 확인**합니다.

### 🧪 연습 4 — AI와 함께
AI에게 이렇게 물어보세요:
> "파이썬 정규표현식으로, 'CO2 612', 'CO2 845.5' 처럼 **CO2 뒤에 오는 숫자만** 뽑는 패턴을 알려 줘. 예시 문자열: '1층 CO2 612, 2층 CO2 845.5, 폼알 45.2'"

받은 패턴으로 `612`, `845.5` 만 나오는지 확인하세요.

In [ ]:
import re
memo2 = "1층 CO2 612, 2층 CO2 845.5, 폼알 45.2"
print(re.findall(r"CO2\s*(\d+\.?\d*)", memo2))    # ['612', '845.5']

---
## 정리
| 하고 싶은 일 | 방법 |
|---|---|
| 앞 10자리 | `s[:10]` |
| 기호로 나누기 | `s.split("-")` → 리스트 |
| 공백·글자 바꾸기 | `strip()`, `replace(" ", "")` |
| 이름 통일 | **표준 이름 사전** + `.get()` |
| 패턴 찾기 | `re.findall(패턴, s)` — AI로 만들고 **반드시 테스트** |

**다음 시간:** 19회 파일과 폴더 — 폴더 안의 CSV 파일들을 코드로 읽고 정리하기